# Beginner tour 7: Scaling up, from 21,000 molecules to 100 million

So far everything ran on 21,044 molecules in seconds. The real project runs the **same loop**
on pools of 2.1 million (ENHITS), 99 million (AmpC) and 116 million (D4) molecules, and has
built a 1.5-billion-molecule pool. Most of the code you haven't seen yet exists only because
of size. This notebook explains it, with real measurements:

1. How much memory the data needs: simple arithmetic
2. Reading files bigger than memory: **memory-mapping**
3. Why **reading order** matters so much (tens to hundreds of times slower)
4. Working in **chunks**
5. Splitting the work across many GPUs: **shards** and **marker files**
6. Asking a cluster for computers: **Slurm**
7. How the embeddings themselves get made: the `embed/` folder

About 2 minutes in total.

In [ ]:
import os, time, tempfile
from pathlib import Path
import numpy as np
from tour import *

def gb(n_bytes):
    return f"{n_bytes / 1e9:,.1f} GB"
print("ready")

---
## 1. How much memory? Just multiply

An embedding table is rows × columns numbers, and each number (`float32`) takes **4 bytes**.
So a table needs molecules × numbers-per-molecule × 4 bytes.

🐍 **Python corner: underscores in numbers.** `2_104_318` is the same as `2104318`. Python lets
you write `_` between digits so big numbers are readable.

In [ ]:
widths = {"molformer": 768, "unimol2": 768, "mhgged": 1024, "grover": 3400, "smited": 768}
total_width = sum(widths.values())           # all five backbones side by side

pools = {"our mini pool": 21_044, "ENHITS": 2_104_318, "AmpC": 99_459_561, "D4": 116_241_184, "Enamine REAL": 1_530_000_000}
for name, n in pools.items():
    print(f"{name:14s} {n:>13,} molecules -> all 5 backbones: {gb(n * total_width * 4):>12}")

Check the arithmetic against a real file on disk:

In [ ]:
real = Path("/N/project/SingleCell_Image/mengjing/enhits_large/embed/molformer_embeddings.npy")
print("predicted:", gb(2_104_318 * 768 * 4), "   actual file:", gb(os.path.getsize(real)))

A typical laptop has 16 GB of memory. The computer running this notebook is a big cluster node:

In [ ]:
with open("/proc/meminfo") as f:
    mem_kb = int(f.readline().split()[1])     # first line: "MemTotal: <number> kB"
print("this computer's memory:", gb(mem_kb * 1000))

Even this machine couldn't hold AmpC's or D4's embeddings in memory all at once, let alone
copies of them (and many NumPy operations quietly make copies, as section 3 shows). So the
code must **never load everything at once**.

---
## 2. Memory-mapping: use a file as if it were in memory

`np.load(path, mmap_mode="r")` doesn't read the file. It returns an array that **reads from
disk only the parts you actually touch**. This is called a **memory map**. Opening a 6.5 GB
file this way takes under a second, however big the file is:

In [ ]:
t = time.time()
big = np.load(real, mmap_mode="r")
print(f"opened {big.shape} in {time.time() - t:.3f} s   (type: {type(big).__name__})")
print("row 1,000,000, first 4 numbers:", np.asarray(big[1_000_000, :4]).round(3))

The project's loader, `EmbeddingFeaturizer.load()` in `molpal/featurizer.py`, opens every
`.npy` embedding file exactly like this. That's why runs start in seconds even on huge pools.

---
## 3. Reading order matters, a lot

A memory map reads from disk on demand. Disks (especially the shared network storage on a
cluster) are fast at reading **one long stretch** and slow at **jumping around**. Let's
measure on a real 6.5 GB file. We read 500 rows that sit next to each other, then 500 rows
scattered at random.

(This cell takes about 10 seconds. It picks new random rows every time it runs, so the
computer can't cheat by remembering rows it read before.)

In [ ]:
smited = np.load("/N/project/SingleCell_Image/mengjing/enhits_large/embed/smited_embeddings.npy", mmap_mode="r")
n = 500
start = np.random.default_rng().integers(0, len(smited) - n)      # a random place to start
t = time.time(); np.array(smited[start:start + n]); t_together = time.time() - t

rows = np.sort(np.random.default_rng().choice(len(smited), n, replace=False))
t = time.time(); smited[rows]; t_scattered = time.time() - t

print(f"{n} rows next to each other: {t_together:.3f} s")
print(f"{n} rows scattered:          {t_scattered:.3f} s   ({t_scattered / max(t_together, 1e-4):,.0f}x slower)")

Same amount of data, many times slower. The exact factor varies with how busy the shared
storage is: measured while writing this notebook, it ranged from about 45× to 230×. This is
the real story behind the long comment in
`EmbeddingMVEModel._get_X` (`molpal/models/mvemodels.py`). On the 99-million-molecule AmpC
pool, workers sat stuck for minutes on scattered reads. The fix was to always read **one
contiguous slice** and patch the few rows that don't fit.

🐍 **Python corner: slices vs index lists.** `table[a:b]` (a **slice**) takes a stretch and is
a **view**: no copying, the same memory seen through a window. `table[[5, 9, 2]]` (an **index
list**) builds a brand-new **copy**. At 100 million rows, accidental copies are how programs
run out of memory.

In [ ]:
small = np.arange(12).reshape(6, 2)
view = small[1:3]
copy = small[[1, 2]]
print("slice shares memory with the original? ", np.shares_memory(small, view))
print("index list shares memory?              ", np.shares_memory(small, copy))
view[0, 0] = 99
print("after changing the view, the original row 1 is:", small[1])

---
## 4. Working in chunks

Each round, the model must guess **every** undocked molecule. On 2 million molecules that's
done in **chunks** of 50,000: load a chunk, guess, keep only the guesses, move on. That's
`_chunked_get_means_and_vars` in `run_experiment.py`:

In [ ]:
import inspect, run_experiment as rx
print("chunk size:", rx.POOL_PREDICT_CHUNK)
src = inspect.getsource(rx._chunked_get_means_and_vars)
print(src[src.index("    surrogate = getattr"):])

The `for start in range(0, len(pool_smi), POOL_PREDICT_CHUNK)` loop is the pattern. Here it
is on the real 6.5 GB file: we compute the average of every number in the whole file while
only ever holding 50,000 rows (150 MB) in memory.

🐍 **Python corner: `range(start, stop, step)`** counts `start, start+step, ...` up to (not
including) `stop`. Perfect for walking through something in steps.

In [ ]:
chunk = 50_000
total, count = 0.0, 0
t = time.time()
for start in range(0, 500_000, chunk):            # the first 500k rows, to keep it quick
    block = np.asarray(big[start:start + chunk])  # contiguous: fast
    total += block.sum(dtype=np.float64)
    count += block.size
print(f"mean of {count:,} numbers = {total / count:.5f}, in {time.time() - t:.1f} s, never more than {gb(chunk * 768 * 4)} in memory")

A subtle trap the project hit here, covered in notebook 8: some models **rank** molecules,
and a rank computed inside one chunk ("best in this chunk") isn't the same as a rank over the
whole pool. That's why the code above has a special path (`hasattr(surrogate, "combine")`)
that collects every chunk first and ranks once.

---
## 5. Many GPUs at once: shards and marker files

Even in chunks, predicting 99 million molecules on one GPU takes hours per round. So the
`--parallel-predict` mode splits the pool into **shards** (fixed slices), one per GPU job:

* **one orchestrator** trains the small model on the docked molecules (cheap) and saves it to a file
* **N workers** (`predict_pool_shard_worker.py`), each owning one shard, load that model and predict their slice
* they coordinate through **files** in a shared folder, not over a network:

```
orchestrator writes   round_1_surrogate.pt, then round_1_ready.marker
worker 3 sees the ready marker, predicts its rows, writes round_1_shard_3_mu.npy, then round_1_shard_3.done
orchestrator waits until every shard's .done exists, joins the results, picks the batch
... next round ...    and at the end: STOP.marker
```

Why files? Cluster jobs start at different times, on different machines, and can crash.
Files on shared storage are the simplest thing they can all see. The "write the data first,
then the marker" order means a marker never points at a half-written file.

How are shards cut? `_chunk_bounds` in `smiles_chunking.py`, the exact function the real
workers use:

In [ ]:
from smiles_chunking import _chunk_bounds
n_pool, n_shards = 99_459_561, 8
for i in range(n_shards):
    s, e = _chunk_bounds(n_pool, i, n_shards)
    print(f"shard {i}: rows {s:>11,} to {e:>11,}  ({e - s:,} molecules)")

The course has a runnable miniature of the whole protocol: 4 worker processes, an
orchestrator, and real marker files, on 40,000 fake molecules. Let's run it (about 45 seconds):

In [ ]:
!cd ../06_al_molecular_tour && python mini_shard_workers.py 2>&1 | tail -12

Look at the order of the worker lines: each round they finish in a **different** order,
because they really run at the same time. The orchestrator doesn't care. It just waits for
all the `.done` files.

---
## 6. Asking the cluster for computers: Slurm

You don't run 8 GPUs from a notebook. University clusters use a **scheduler** called
**Slurm**: you write a small script saying what you need, submit it with `sbatch script.sh`,
and Slurm runs it when the machines are free. The settings are lines starting with
`#SBATCH`. Here are the ones from the project's main script:

In [ ]:
script = AL_REPO / "slurm/al_runs/enamine/submit_al_runs.sh"
for line in script.read_text().splitlines():
    if line.startswith("#SBATCH") and "mail" not in line:
        print(line)

| Line | Means |
|---|---|
| `-J molpal_al` | the job's name |
| `-p hopper`, `-q hopper` | which group of machines (here: the ones with H100 GPUs) |
| `--gpus-per-node h100:1` | one H100 GPU |
| `--cpus-per-task=8` | 8 CPU cores |
| `--time=24:00:00` | stop after at most 24 hours |
| `--mem=0` | all the memory on the machine |
| `-o logs/...txt`, `-e logs/...err` | where printed output and errors go (`%A_%a` is replaced by the job number) |
| `-A r00939` | which allocation (account) pays for the compute time |

The `slurm/` folder holds one such script per kind of experiment, organized by dataset
(`enamine`, `ampc`, `d4`).

---
## 7. Where embeddings come from: `embed/`

Before any active learning, every molecule must be run through every backbone, once. On
100 million molecules that's the most expensive step of all: GPU-days per backbone. So it
is also chunked:

1. `embed/compute/compute_<backbone>_embeddings_chunk.py --chunk-id i --num-chunks K` embeds one
   slice of the SMILES list and writes it to its own file. Hundreds of these run as one Slurm
   "array" job.
2. `embed/stitch/stitch_embedding_chunks.py` joins the slices into one `(N, D)` `.npy` file.

In [ ]:
for folder in ["embed/compute", "embed/stitch"]:
    names = sorted(p.name for p in (AL_REPO / folder).glob("*.py"))
    print(f"{folder}: {len(names)} scripts, e.g. {names[:4]}")

There's a neat trick for the joining step in `shared_embedding_store.py`. **Preallocate** the
full-size file once (empty), then let each chunk job **write its slice straight into place**.
Here it is with the real functions, on a small file:

In [ ]:
from shared_embedding_store import preallocate, write_slice

path = Path(tempfile.mkdtemp()) / "demo_embeddings.npy"
n_mol, width, n_chunks = 1_000, 8, 4
preallocate(path, n_molecules=n_mol, dim=width)                     # a full-size, empty file
for chunk_id in range(n_chunks):                                     # really: 4 separate Slurm jobs
    s, e = _chunk_bounds(n_mol, chunk_id, n_chunks)
    fake_embeddings = np.full((e - s, width), chunk_id, dtype=np.float32)   # stand-in for a backbone's output
    write_slice(path, s, e, fake_embeddings)

final = np.load(path, mmap_mode="r")
print("shape:", final.shape, "  rows 0, 300, 600, 999 came from chunks:", final[[0, 300, 600, 999], 0])

The real backbones (`grover/`, `unimol1/`, `unimol2/`, `ibm_materials/`, `muben/`) are large
model libraries that the `embed/` scripts call. You can treat them as black boxes:
**SMILES in, embedding out**.

---
## Summary

| Problem at scale | Where in the code | The idea |
|---|---|---|
| the data is bigger than memory | `EmbeddingFeaturizer.load` | **memory-map**: read only what you touch |
| scattered reads are tens to hundreds of times slower | `EmbeddingMVEModel._get_X` | read **contiguous** slices, patch the exceptions |
| copies blow up memory | throughout | prefer **slices** (views) over index lists (copies) |
| predicting the whole pool every round | `_chunked_get_means_and_vars` | **chunks** of 50,000 |
| one GPU is too slow | `ParallelMVEExplorer`, `predict_pool_shard_worker.py` | **shards** coordinated by **marker files** |
| getting machines | `slurm/` | **Slurm** job scripts, `sbatch` |
| making embeddings at all | `embed/compute`, `embed/stitch`, `shared_embedding_store.py` | chunk jobs + **preallocate and write in place** |

🐍 **Python you met:** `_` in numbers, `os.path.getsize`, reading `/proc/meminfo`, memory
maps, timing with `time.time()`, views vs copies, `np.shares_memory`, `range` with a step,
running scripts with `!`.

## Try it yourself
1. In section 3, change `n = 500` to `n = 50`. Is scattered reading still slower? (Even small numbers of jumps cost.)
2. In section 4, how long would the whole 2.1M-row file take at the speed you measured?
3. In section 5, how many shards would keep each shard under 5 million molecules for D4 (116 million)?

**Next:** `8_tests_and_bugs.ipynb`, how the project caught three real bugs, and how tests keep them from coming back.